# Download audio (pytubefix)
Backup if yt-dlp doesn't work. Saves `<video_id>.wav` to `data/raw_audios/`.

In [ ]:
import re
import subprocess
import tempfile
from pathlib import Path

from pytubefix import YouTube

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
out_dir = root / "data" / "raw_audios"
out_dir.mkdir(parents=True, exist_ok=True)

links = re.findall(r"https?://[^\s)>]+", (root / "data" / "podcasts.md").read_text(encoding="utf-8"))
print(len(links), "links")

In [ ]:
def download(link):
    yt = YouTube(link)
    wav = out_dir / f"{yt.video_id}.wav"
    if wav.exists():
        print("exists:", wav.name)
        return
    stream = yt.streams.get_audio_only()
    if stream is None:
        raise ValueError("no audio stream")
    with tempfile.TemporaryDirectory(dir=out_dir) as tmp:
        src = stream.download(output_path=tmp, filename="audio")
        tmp_wav = Path(tmp) / "audio.wav"
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-vn", "-acodec", "pcm_s16le", str(tmp_wav)], check=True)
        tmp_wav.rename(wav)
    print("saved:", wav.name)


failed = []
for i, link in enumerate(links, 1):
    print(f"[{i}/{len(links)}] {link}")
    try:
        download(link)
    except Exception as e:
        failed.append(link)
        print("failed:", e)
print(f"done, {len(failed)} failed")